# ACT training on Kaggle (SO-101 hand_tracking_pick_place)

Twin of the Diffusion-Policy notebook — **same dataset, same robot, same deploy script** — so the
two policies are directly comparable. Only the policy type + its knobs differ.

**Before running — Notebook settings (right panel):**
- **Accelerator:** GPU **T4 x2** (use T4, **not P100** — Kaggle's torch dropped sm_60; P100 fails with
  "not compatible with the current PyTorch installation").
- **Internet:** **On** (pip + HF dataset/push)
- **Add-ons -> Secrets:** secret named **`HF_TOKEN`** = your Hugging Face *write* token

Workflow: run cells 1-4 interactively to validate (STEPS=2000), check it/s, then set the real STEPS and
use **Save Version -> Save & Run All (Commit)** to run headless to completion.

**Why ACT here:** it works in the ~100-demo regime and its **temporal ensembling** directly targets the
in-air jitter we saw on the DP eval. ACT is lighter than DP (no diffusion U-Net / denoising loop), so it
trains faster per step.

In [ ]:
# 1. Install LeRobot and the grip-aux policy plugin.
!pip install -q "lerobot[training] @ git+https://github.com/huggingface/lerobot@da92db8fc0c935950a56b1ea61fa9b211ef3ac30"
!pip install -q torchcodec
!pip install -q "huggingface_hub==1.19.0" "transformers==5.5.4"

# Upload webcam-input/.../lerobot_policy_grip_aux as a Kaggle Dataset and attach it as an Input.
import glob, os
_plugin = glob.glob("/kaggle/input/**/lerobot_policy_grip_aux/pyproject.toml", recursive=True)
if not _plugin:
    raise RuntimeError("Attach a Kaggle input containing the lerobot_policy_grip_aux source directory")
PLUGIN_ROOT = os.path.dirname(_plugin[0])
!pip install -q --no-deps --no-build-isolation {PLUGIN_ROOT}
print("grip-aux plugin:", PLUGIN_ROOT)

In [ ]:
# 2. Log in to Hugging Face from the Kaggle secret
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
# 3. Sanity check GPU + torchcodec
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))
try:
    import torchcodec; print("torchcodec OK", torchcodec.__version__)
except Exception as e:
    print("torchcodec NOT available -> remove the --dataset.video_backend flag below.", e)

In [ ]:
# 4. Train.  ====== EDIT THESE KNOBS ======
STEPS             = 50000   # 2000 = validation/speed check. Real run: ~100000 (1 GPU) or ~60000 (2 GPUs).
CHUNK             = 50     # action-chunk length. The model predicts CHUNK actions per query.
                           # ~= 5 s at our 10 fps (ACT's 100 default assumed ~50 Hz; we record 10 fps).
TEMPORAL_ENSEMBLE = True   # ACT's jitter smoother: re-query EVERY step + exp-weighted average (coeff 0.01)
                           # of overlapping chunk predictions. This is the main reason to try ACT here.
                           # It only affects INFERENCE, but we bake it into the saved config so
                           # deploy_so101_ee.py picks it up automatically (no deploy code change).
USE_BOTH_GPUS     = False  # True = Kaggle T4 x2 via accelerate (~2x). Requires Accelerator: GPU T4 x2.
BATCH             = 32     # per-GPU batch. ACT is lighter than DP -> you can raise to 64 on a T4.
# =========================================
# Notes:
#   * LR is left at ACT's default 1e-5 (the ACT-paper value, lerobot's default) - don't override.
#   * use_vae stays True (CVAE - the latent that lets ACT model multi-modal human demos).
#   * Validation requires n_obs_steps=1 (ACT default) and, with temporal ensembling, n_action_steps=1.
#   * Budget 12h = 43200s. ACT is faster/step than DP; STEPS=100000 typically fits on 1 GPU, and on
#     ~100 demos ACT usually converges well before that - watch the loss and stop early if flat.

PUSH = (STEPS >= 50000)    # only push a real (long) run to the Hub

# Shared training args (NOT including the launcher or the AMP flag, which differ per path).
args = (
    " --policy.type=grip_aux_act"
    " --dataset.repo_id=stevenzenith/hand_tracking_pv_pick_place"
    " --dataset.video_backend=torchcodec"
    f" --batch_size={BATCH}"     # per-GPU batch (global = BATCH x num_gpus)
    " --num_workers=4"
    f" --policy.chunk_size={CHUNK}"
    " --policy.grip_aux_weight=0.25"
    + (" --policy.n_action_steps=1 --policy.temporal_ensemble_coeff=0.01"
       if TEMPORAL_ENSEMBLE else f" --policy.n_action_steps={CHUNK}")
    + f" --steps={STEPS}"
    " --save_freq=10000"
    " --output_dir=/kaggle/working/act_pv_grip"
    " --job_name=act_pv_grip"
    " --policy.device=cuda"
    f" --policy.push_to_hub={str(PUSH).lower()}"
    " --policy.repo_id=stevenzenith/act_pv_grip"
    " --wandb.enable=false"
)

# expandable_segments avoids allocator fragmentation under AMP (fp16/fp32 buffers -> the
# "reserved but unallocated" OOM on the 15GB T4).
ALLOC = "PYTORCH_ALLOC_CONF=expandable_segments:True "
if USE_BOTH_GPUS:
    cmd = ALLOC + "accelerate launch --multi_gpu --num_processes=2 --mixed_precision=fp16 $(which lerobot-train)" + args
else:
    cmd = ALLOC + "lerobot-train --policy.use_amp=true" + args

import glob as _glob
if _glob.glob("/kaggle/input/**/checkpoints/[0-9]*", recursive=True):
    print("Checkpoint input detected -> SKIPPING fresh training. Run cell 4b to RESUME.")
else:
    print(cmd)
    get_ipython().system(cmd)

## 4b. Resume a cancelled run (from the last saved checkpoint)

Use this **instead of cell 4** when a previous commit hit the 12 h limit. It picks up from the
latest saved checkpoint's optimizer + step state and trains the remaining steps to the original
`--steps` target (no knob changes needed — `chunk_size`, temporal-ensemble, dataset, etc. are read
back from the checkpoint's `train_config.json`). Pushes to the Hub at the end exactly as a fresh run.

**Before running:** Run cells 1-3 first, then **Add Input** (right panel) -> this kernel's committed
**Notebook Output** (or a Dataset you made from it) so the checkpoints mount read-only under
`/kaggle/input/`. This cell auto-finds the highest `checkpoints/NNNNNN/` there, copies it into the
writable `output_dir`, rebuilds the `last` symlink (Kaggle's output download breaks it), and resumes.


In [ ]:
# 4b. RESUME from the last saved checkpoint of a cancelled run.  (Run cells 1-3 first.)
import os, glob, shutil

DST = "/kaggle/working/act_pv_grip"   # must match output_dir baked in the checkpoint's train_config

# Find the highest numbered checkpoint among the mounted Inputs.
ckpts = sorted(
    glob.glob("/kaggle/input/**/checkpoints/[0-9]*", recursive=True),
    key=lambda p: int(os.path.basename(p)),
)
if not ckpts:
    print("Mounted inputs:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "NONE")
    for r, d, f in os.walk("/kaggle/input"):
        print("  ", r)
    raise SystemExit(
        "No checkpoint found under /kaggle/input. Right panel -> Add Input -> Datasets -> "
        "search 'act-pickplace-ckpt-30k' (zhuokaiyuan/act-pickplace-ckpt-30k) and add it, then re-run."
    )
SRC_CKPT = ckpts[-1]                              # e.g. .../checkpoints/030000
SRC_ROOT = SRC_CKPT.split("/checkpoints/")[0]     # the dir that CONTAINS checkpoints/
step = os.path.basename(SRC_CKPT)
print("Found checkpoint:", SRC_CKPT, "| step", step)

# Copy into a writable location (lerobot writes new checkpoints back into output_dir; input is read-only).
if not os.path.exists(DST):
    shutil.copytree(SRC_ROOT, DST)

# Rebuild the 'last' symlink -- Kaggle's output/dataset packaging flattens it; lerobot resumes from it.
last = os.path.join(DST, "checkpoints", "last")
if os.path.islink(last) or os.path.exists(last):
    os.remove(last)
os.symlink(step, last)                            # relative link -> the numbered checkpoint dir
print("Prepared", DST, "| last ->", step)

# --resume=true reloads optimizer/scheduler/step/RNG and continues to the original --steps target.
# --save_freq=4000 checkpoints more often so a second time-limit kill costs less progress.
cmd = ("PYTORCH_ALLOC_CONF=expandable_segments:True "
       f"lerobot-train --config_path={last}/pretrained_model/train_config.json"
       " --resume=true --save_freq=4000")
print(cmd)
get_ipython().system(cmd)


## Budget / what to expect (ACT)
- ACT has **no diffusion denoising loop**, so it's compute-lighter than DP per step - expect a higher
  step/s than the DP run on the same T4 setup. Fill in your measured `step/s` after the 2000-step check.
- Start with the 2000-step run and compare both the existing action loss and `grip_aux_mae`; only a
  held-out improvement justifies enabling low-level grip control.
- **Resume across sessions** (if 12h isn't enough): re-add this notebook's committed output as input and
  add `--resume=true` pointing at the restored `--output_dir`.

## Deploy
Install the same `lerobot_policy_grip_aux` plugin on the robot machine, then deploy the saved 6D policy:
```
./run_deploy_ee.sh --policy stevenzenith/act_pv_grip --grip-context soft
```
PV is not a deployment observation. The policy receives RGB, robot state and the explicit context only.

## Fair comparison checklist (vs DP)
- Use the same new PV-labelled dataset (`stevenzenith/hand_tracking_pv_pick_place`) for both policies.
- Eval both on the same paired object placements and report action success plus held-out grip auxiliary MAE.